# Joint fitting of multiple observations

Different observations may constrain the same source, or only some of its physical parameters. Prism fits them with one objective while retaining each observation's grid, errors and model. Reuse a model to share all its parameters, or link selected parameters between distinct models. Responses can be added when the instruments differ.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import Table

from prism.modeling import Component, select_line
from prism.modeling.lines import GaussianLines
from prism.modeling.fitting import Dataset, FitResult, TRF
from prism.modeling.operators import InstrumentResponse, SpectralResponse

The continuum and line set form the intrinsic source model: the emitted spectrum before instrumental effects. A response transforms it into the spectrum recorded by an instrument.

In [ ]:
def continuum(x, amplitude, slope):
    return amplitude + slope*(x-5000.)/100.

host = Component(continuum, {'amplitude': 1., 'slope': 0.}, name='continuum')
host.amplitude.bounds = (0., 5.)
transitions = {'name': ['Hb', 'OIII'], 'position': np.array([4861.3, 5006.8])*u.AA}
lines = GaussianLines(transitions, amplitude=2., offset=0., fwhm=450., name='lines',
                      bounds={'amplitude': (0., 20.), 'offset': (-300., 300.),
                              'fwhm': (100., 1000.)}, normalization='amplitude')
source = host + lines

Build the synthetic sources before fitting, leaving the starting model unchanged.

In [ ]:
truth_a = source.copy()
truth_a.components['continuum'].amplitude = 1.5
truth_a.components['continuum'].slope = .15
truth_a.components['lines'].amp_hb = 4.
truth_a.components['lines'].amp_oiii = 7.
truth_a.components['lines'].offset = -50.
truth_a.components['lines'].fwhm = 280.

truth_b = truth_a.copy()
truth_b.components['continuum'].amplitude = 2.
truth_b.components['lines'].amp_hb = 5.
truth_b.components['lines'].amp_oiii = 8.

Each response retains its native grid. Instrumental broadening is applied here; profile `instfwhm` remains zero. See [Operators and Responses](Operators%20and%20Responses.ipynb) for the operator and `instfwhm` alternatives.

In [ ]:
grid_a = np.linspace(4800., 5100., 1201)
grid_b = np.linspace(4820., 5080., 1001)
response_a = SpectralResponse(
    instrument=InstrumentResponse.from_fixed_fwhm(grid_a, 1.5), wave=grid_a)
response_b = SpectralResponse(
    instrument=InstrumentResponse.from_fixed_fwhm(grid_b, 2.5), wave=grid_b)

`Dataset` groups one observation's coordinates, values and errors, plus an optional mask and response. It is useful whenever the observation setup should stay separate from the intrinsic model. Labels A and B identify observations, not wavelength bands.

In [ ]:
dataset_a = Dataset(grid_a[1:-1:2], np.zeros(600), yerr=np.full(600, .05), response=response_a)
dataset_b = Dataset(grid_b[1:-1:3], np.zeros(333), yerr=np.full(333, .05), response=response_b)
rng = np.random.default_rng(41)
dataset_a.y = dataset_a.predict(truth_a) + rng.normal(0., dataset_a.yerr)
dataset_b.y = dataset_b.predict(truth_b) + rng.normal(0., dataset_b.yerr)

Distinct source objects have independent parameters. Link only the kinematics that the observations should estimate together.

In [ ]:
source_b = source.copy()
source_b.components['lines'].offset.link = source.components['lines'].offset
source_b.components['lines'].fwhm.link = source.components['lines'].fwhm
observations = {'A': (source, dataset_a), 'B': (source_b, dataset_b)}

fitter = TRF()
result = fitter.jointfit(observations)
{key: result.fit_info[key] for key in ('success', 'stat', 'dof')}

To estimate exactly the same source parameters from both observations, use the same source object in both pairs. Each Dataset still keeps its own response.

In [ ]:
parameters = result['A'].params
Table(rows=[(name, p.value, p.std) for name, p in parameters.items()],
      names=('parameter', 'value', 'std'))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
for ax, label in zip(axes, ('A', 'B')):
    data = result[label].dataset
    ax.plot(data.x, data.y, '.', color='0.6', markersize=2)
    x_plot = np.linspace(data.x.min(), data.x.max(), 1500)
    ax.plot(x_plot, result[label].predict(x_plot), label=label)
    ax.set(xlabel='Wavelength [Angstrom]', ylabel='Flux density')
    ax.legend()

The line profile is intrinsic. Its uncertainty uses the complete fitted covariance, including correlations with the shared kinematics.

In [ ]:
selected = select_line(result['B'], 'OIII', components='lines')
measurement = selected.sample(method='covariance', n_samples=256, random_state=41)
measurement.flux

A fit session stores the associations, data and covariance. Fixed response matrices are companion FITS files and should be moved with the session. Supply the continuum function explicitly when loading; built-in line profiles restore themselves.

In [ ]:
from pathlib import Path

Path('results').mkdir(exist_ok=True)
result.save('results/joint.fits', overwrite=True)
restored = FitResult.load('results/joint.fits', components={'continuum': host})
restored

An association maps an observation label to `(model, Dataset)`. `result.associations` copies the whole mapping together, preserving shared nodes and links. Edit this copy for a new fit without changing the recorded result.

In [ ]:
observations = restored.associations
observations['B'][0].components['continuum'].slope.fixed = True
updated = fitter.jointfit(observations)
{key: updated.fit_info[key] for key in ('success', 'stat', 'dof')}

For independent estimates, reuse the source as a recipe with `multifit`.
Each Dataset gets its own parameters and covariance, including the line kinematics.
The returned dictionary contains ordinary single-fit results.

In [ ]:
independent = fitter.multifit(source, {'A': dataset_a, 'B': dataset_b}, progress=False)
{label: fit.model.components['lines'].fwhm.value for label, fit in independent.items()}